# 🔮 Temporal Fusion Transformer — Financial Time Series Forecasting

**Chạy được cả LOCAL lẫn Google Colab.**

| # | Kỹ thuật | Method |
|---|----------|--------|
| 1 | Optuna HPO | Auto-tune hidden_size, lr, dropout, attention_heads |
| 2 | Walk-Forward CV | Time-aware cross-validation, không data leakage |
| 3 | Quantile Forecasting | Dự báo q02/q10/q25/q50/q75/q90/q98 |
| 4 | Checkpoint Ensemble | Average top-3 best checkpoints |

## 📦 Section 0 — Cài đặt dependencies

In [39]:
import subprocess, sys

pkgs = [
    'pytorch-forecasting>=1.0.0',
    'lightning>=2.0.0',
    'torch>=2.0.0',
    'optuna>=3.0.0',
    'plotly>=5.0.0',
    'scikit-learn>=1.3.0',
]
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('Cai dat hoan tat')

Cai dat hoan tat


In [40]:
import torch
device = 'GPU' if torch.cuda.is_available() else 'CPU (training se cham hon)'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: CPU (training se cham hon)


## 📁 Section 1 — Tự động nhận diện môi trường & import

In [41]:
import sys, os
from pathlib import Path

# ── Tự động nhận diện LOCAL vs COLAB ─────────────────────────────────────────
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

print(f'Moi truong: {"Google Colab" if IS_COLAB else "Local"}')

if IS_COLAB:
    PROJECT_DIR = '/content'
    DATA_DIR    = '/content'
else:
    _nb_dir = Path(globals().get('__vsc_ipynb_file__',
                   Path.cwd() / 'notebooks' / 'tft_colab.ipynb')).parent
    PROJECT_DIR = str(_nb_dir.parent)
    DATA_DIR    = str(_nb_dir.parent / 'colab_data')

print(f'PROJECT_DIR = {PROJECT_DIR}')
print(f'DATA_DIR    = {DATA_DIR}')

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Verify
assert Path(PROJECT_DIR, 'models', 'tft_model.py').exists(), \
    f'Khong tim thay models/tft_model.py tai {PROJECT_DIR}'

Moi truong: Local
PROJECT_DIR = c:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection
DATA_DIR    = c:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection\colab_data


In [42]:
# Tự động reload module nếu kernel chưa restart
import importlib, sys
for mod_name in ['models.tft_model', 'models']:
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])

from models.tft_model import TFTConfig, TFTForecaster
print('Import TFTForecaster thanh cong')

Import TFTForecaster thanh cong


## ⚙️ Section 2 — Cấu hình

In [44]:
TRAIN_JSON = os.path.join(DATA_DIR, 'train.json')
TEST_JSON  = os.path.join(DATA_DIR, 'test.json')

assert Path(TRAIN_JSON).exists(), (
    f'Khong tim thay {TRAIN_JSON}.\n'
    'LOCAL: Chay truoc: python scripts/export_for_colab.py'
)
print(f'train.json: {TRAIN_JSON} OK')
print(f'test.json : {TEST_JSON} OK')

OUTPUT_DIR = os.path.join(PROJECT_DIR, 'tft_output')

cfg = TFTConfig(
    target='close',
    group_col='symbol',
    time_col='open_time',

    max_encoder_length=60,
    max_prediction_length=7,

    known_real_features=[
        'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
        'month_sin', 'month_cos', 'doy_sin', 'doy_cos',
    ],

    unknown_real_features=[
        'return_pct', 'log_return',
        'rsi_14', 'macd_hist', 'macd', 'macd_signal',
        'bb_pct', 'bb_width', 'atr_14', 'volume_ratio',
        'rolling_vol_30', 'price_zscore', 'drawdown_pct',
        'ma_7', 'ma_25', 'ema_12', 'ema_26',
    ],

    quantiles=[0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98],

    batch_size=64,
    max_epochs=30,
    early_stopping_patience=10,
    val_ratio=0.15,

    learning_rate=0.03,
    hidden_size=64,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=16,

    n_optuna_trials=20,
    n_cv_folds=5,

    output_dir=OUTPUT_DIR,
)

print(f'Config OK | output -> {OUTPUT_DIR}')

train.json: c:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection\colab_data\train.json OK
test.json : c:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection\colab_data\test.json OK
Config OK | output -> c:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection\tft_output


## 📂 Section 3 — Load data

In [45]:
import pandas as pd
import numpy as np

forecaster = TFTForecaster(cfg)

df_train, df_val, df_test = forecaster.load_and_split(
    train_path=TRAIN_JSON,
    test_path=TEST_JSON,
    val_ratio=cfg.val_ratio,
    symbol='BTC/USDT',
)

print(f'df_train : {df_train.shape}')
print(f'df_val   : {df_val.shape}')
print(f'df_test  : {df_test.shape}')
print(f'Columns: {list(df_train.columns)[:10]} ...')

10:21:14 [INFO] TFTForecaster ready | target=close | encoder=60 | pred_len=7 | quantiles=[0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98]
10:21:14 [INFO] Loading train: c:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection\colab_data\train.json
10:21:14 [INFO] Loading test:  c:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection\colab_data\test.json
10:21:14 [INFO] Split complete | train=512 | val=91 | test=107


df_train : (512, 48)
df_val   : (91, 48)
df_test  : (107, 48)
Columns: ['open_time', 'open', 'high', 'low', 'close', 'volume', 'return_pct', 'log_return', 'ma_7', 'ma_25'] ...


In [46]:
print('Train sample (tail):')
display(df_train.tail(3))
print(f'\nTarget stats: {df_train[cfg.target].describe()}')

Train sample (tail):


,open_time,open,high,low,close,volume,return_pct,log_return,ma_7,ma_25,...,dow_sin,dow_cos,month_sin,month_cos,dom_sin,dom_cos,doy_sin,doy_cos,symbol,time_idx
509,2026-01-28 00:00:00+00:00,89249.99,90600.00,88833.65,89299.99,14332.18502,0.056011,0.000560,88850.385714,91594.6364,...,0.974928,-0.222521,0.0,1.0,-0.724793,0.688967,0.447945,0.894061,BTC/USDT,509
510,2026-01-29 00:00:00+00:00,89300.00,89348.00,83383.33,84650.16,30431.38201,-5.206977,-0.053474,88149.027143,91319.4536,...,0.433884,-0.900969,0.0,1.0,-0.571268,0.820763,0.463258,0.886224,BTC/USDT,510
511,2026-01-30 00:00:00+00:00,84650.16,84735.75,81118.00,84260.49,35195.84793,-0.460330,-0.004614,87386.202857,90935.4848,...,-0.433884,-0.900969,0.0,1.0,-0.394356,0.918958,0.478434,0.878124,BTC/USDT,511



Target stats: count       512.000000
mean      95749.419922
std       15902.314084
min       53962.970000
25%       87314.687500
50%       96557.605000
75%      108195.240000
max      124658.540000
Name: close, dtype: float64


In [47]:
# TFT can encoder history truoc test
encoder_tail = df_train.tail(cfg.max_encoder_length).copy()
df_test_with_history = pd.concat([encoder_tail, df_test], ignore_index=True)
print(f'df_test_with_history: {df_test_with_history.shape}')

df_test_with_history: (167, 48)


## 🔍 Section 4 — [Kỹ thuật 1] Optuna HPO

In [48]:
RUN_OPTUNA = False   # Dat True de auto-tune hyperparameters voi Optuna

if RUN_OPTUNA:
    best_hparams = forecaster.tune_hyperparameters(
        df_train=df_train,
        df_val=df_val,
        n_trials=cfg.n_optuna_trials,
    )
    print('Best hyperparameters:')
    for k, v in best_hparams.items():
        print(f'  {k}: {v}')
else:
    print('Skip Optuna HPO')

Skip Optuna HPO


## 📊 Section 5 — [Kỹ thuật 2] Walk-Forward Cross-Validation

In [49]:
RUN_CV = False   # Dat True de danh gia qua Walk-Forward CV

if RUN_CV:
    cv_results = forecaster.walk_forward_cv(
        df=df_train,
        n_splits=cfg.n_cv_folds,
    )
    valid_folds = [r for r in cv_results if 'val_loss' in r]
    for r in valid_folds:
        print(f"Fold {r['fold']}: val_loss={r['val_loss']:.4f} | train={r['train_rows']} | val={r['val_rows']}")
    if valid_folds:
        losses = [r['val_loss'] for r in valid_folds]
        print(f'Mean val_loss = {np.mean(losses):.4f} +/- {np.std(losses):.4f}')
else:
    print('Skip Walk-Forward CV')

Skip Walk-Forward CV


## 🚂 Section 6 — Final Training

In [50]:
train_result = forecaster.train(df_train=df_train, df_val=df_val)

print('Training xong:')
print(f"  val_loss        = {train_result['val_loss']:.4f}")
print(f"  epochs          = {train_result['n_epochs']}")
print(f"  best_checkpoint = {train_result['best_checkpoint']}")

10:21:31 [INFO] Features — known_reals: 9, unknown_reals: 18, target: close
10:21:31 [INFO] TFT model: 353,122 parameters
10:21:31 [INFO] GPU available: False, used: False
10:21:31 [INFO] TPU available: False, using: 0 TPU cores
10:21:31 [INFO] 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
10:21:31 [INFO] Trainer ready | accelerator=cpu | max_epochs=30
10:21:31 [INFO] ============================================================
10:21:31 [INFO] Starting final TFT training ...
10:21:31 [INFO] ============================================================


Epoch 13: 100%|██████████| 8/8 [00:04<00:00,  1.75it/s, train_loss_step=1.33e+3, val_loss=1.38e+4, train_loss_epoch=1.3e+3] 

10:22:49 [INFO] Training done | val_loss=6525.6880 | epochs=14
10:22:49 [INFO] Best checkpoint: C:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection\tft_output\checkpoints\tft_final_epoch=003_vallossval_loss=6525.6880.ckpt



Training xong:
  val_loss        = 6525.6880
  epochs          = 14
  best_checkpoint = C:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection\tft_output\checkpoints\tft_final_epoch=003_vallossval_loss=6525.6880.ckpt


## 🔮 Section 7 — [Kỹ thuật 3 & 4] Quantile + Ensemble Prediction

In [51]:
predictions = forecaster.ensemble_predict(
    df_test=df_test_with_history,
    n_best=3,
)

print(f'Predictions shape: {predictions.shape}')
print(f'Columns: {list(predictions.columns)}')
display(predictions.head(10))

10:22:49 [INFO] Ensembling 3 checkpoints:
10:22:49 [INFO]   tft_final_epoch=001_vallossval_loss=7861.8687.ckpt
10:22:49 [INFO]   tft_final_epoch=003_vallossval_loss=6525.6880.ckpt
10:22:49 [INFO]   tft_final_epoch=005_vallossval_loss=7090.4102.ckpt
10:22:49 [INFO] Loaded: tft_final_epoch=001_vallossval_loss=7861.8687.ckpt
10:22:50 [INFO] Loaded: tft_final_epoch=003_vallossval_loss=6525.6880.ckpt
10:22:51 [INFO] Loaded: tft_final_epoch=005_vallossval_loss=7090.4102.ckpt
10:22:51 [INFO] Ensemble complete | models=3 | rows=7


Predictions shape: (7, 9)
Columns: ['window', 'step', 'q02', 'q10', 'q25', 'q50', 'q75', 'q90', 'q98']


,window,step,q02,q10,q25,q50,q75,q90,q98
0,0,1,70594.585938,76900.726562,83192.859375,89155.804688,94400.406250,100444.132812,107391.562500
1,0,2,70745.679688,77002.148438,83077.351562,89107.062500,94163.351562,100362.039062,106913.023438
2,0,3,70284.132812,77134.304688,83058.148438,89268.132812,94570.906250,100414.554688,107656.664062
3,0,4,70927.507812,78094.898438,83874.179688,90111.132812,94975.937500,101065.250000,107775.210938
4,0,5,70925.359375,78166.226562,83924.218750,90149.898438,95014.820312,101124.476562,107802.898438
5,0,6,70619.312500,77823.804688,83638.843750,89878.070312,94891.851562,100845.125000,107716.250000
6,0,7,70363.859375,76975.843750,83051.804688,89159.632812,94321.875000,100329.085938,107159.867188


## 📏 Section 8 — Evaluation

In [52]:
y_true = df_test[cfg.target].values

metrics = forecaster.evaluate(y_true=y_true, predictions=predictions)

print(f"MAE    : {metrics['MAE']:.4f}")
print(f"RMSE   : {metrics['RMSE']:.4f}")
print(f"MAPE%  : {metrics['MAPE_%']:.2f}%")
print(f"sMAPE% : {metrics['sMAPE_%']:.2f}%")
if 'Winkler_80' in metrics:
    print(f"Winkler80  : {metrics['Winkler_80']:.4f}")
    print(f"Coverage80%: {metrics['Coverage_80_%']:.1f}%")
if 'Winkler_96' in metrics:
    print(f"Winkler96  : {metrics['Winkler_96']:.4f}")
    print(f"Coverage96%: {metrics['Coverage_96_%']:.1f}%")

10:22:51 [INFO] ── Evaluation Metrics ─────────────────────
10:22:51 [INFO]   MAE                 : 10468.9547
10:22:51 [INFO]   RMSE                : 10468.9547
10:22:51 [INFO]   MAPE_%              : 13.3046
10:22:51 [INFO]   sMAPE_%             : 12.4747
10:22:51 [INFO]   Winkler_80          : 23543.4062
10:22:51 [INFO]   Coverage_80_%       : 100.0000
10:22:51 [INFO]   Winkler_96          : 36796.9766
10:22:51 [INFO]   Coverage_96_%       : 100.0000
10:22:51 [INFO] ───────────────────────────────────────────


MAE    : 10468.9547
RMSE   : 10468.9547
MAPE%  : 13.30%
sMAPE% : 12.47%
Winkler80  : 23543.4062
Coverage80%: 100.0%
Winkler96  : 36796.9766
Coverage96%: 100.0%


## 💾 Section 9 — Lưu kết quả JSON

In [53]:
import json

RESULT_PATH = os.path.join(OUTPUT_DIR, 'tft_results.json')

forecaster.save_results(
    output_path=RESULT_PATH,
    y_true=y_true,
    predictions=predictions,
    train_result=train_result,
    extra={'symbol': 'BTC/USDT'},
)

print(f'Results saved -> {RESULT_PATH}')

with open(RESULT_PATH) as f:
    r = json.load(f)
for k, v in r.items():
    if isinstance(v, dict): print(f'  {k}: dict({len(v)} keys)')
    elif isinstance(v, list): print(f'  {k}: list({len(v)} items)')
    else: print(f'  {k}: {str(v)[:60]}')

10:22:51 [INFO] ── Evaluation Metrics ─────────────────────
10:22:51 [INFO]   MAE                 : 10468.9547
10:22:51 [INFO]   RMSE                : 10468.9547
10:22:51 [INFO]   MAPE_%              : 13.3046
10:22:51 [INFO]   sMAPE_%             : 12.4747
10:22:51 [INFO]   Winkler_80          : 23543.4062
10:22:51 [INFO]   Coverage_80_%       : 100.0000
10:22:51 [INFO]   Winkler_96          : 36796.9766
10:22:51 [INFO]   Coverage_96_%       : 100.0000
10:22:51 [INFO] ───────────────────────────────────────────
10:22:51 [INFO] Results saved → c:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection\tft_output\tft_results.json


Results saved -> c:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection\tft_output\tft_results.json
  config: dict(22 keys)
  best_hparams: dict(0 keys)
  cv_results: list(0 items)
  train_result: dict(3 keys)
  metrics: dict(8 keys)
  predictions: dict(9 keys)
  y_true: list(107 items)
  symbol: BTC/USDT


## 📈 Section 10 — Visualization

In [54]:
import plotly.graph_objects as go
import numpy as np

step1 = predictions[predictions['step'] == 1].reset_index(drop=True)
n_plot = min(len(step1), len(y_true))
x_axis = list(range(n_plot))

fig = go.Figure()

# 96% CI
if 'q02' in step1.columns and 'q98' in step1.columns:
    fig.add_trace(go.Scatter(
        x=x_axis + x_axis[::-1],
        y=step1['q02'][:n_plot].tolist() + step1['q98'][:n_plot].tolist()[::-1],
        fill='toself', fillcolor='rgba(88,166,255,0.08)',
        line=dict(color='rgba(0,0,0,0)'), name='96% CI',
    ))

# 80% CI
if 'q10' in step1.columns and 'q90' in step1.columns:
    fig.add_trace(go.Scatter(
        x=x_axis + x_axis[::-1],
        y=step1['q10'][:n_plot].tolist() + step1['q90'][:n_plot].tolist()[::-1],
        fill='toself', fillcolor='rgba(88,166,255,0.18)',
        line=dict(color='rgba(0,0,0,0)'), name='80% CI',
    ))

fig.add_trace(go.Scatter(
    x=x_axis, y=y_true[:n_plot],
    name='Actual', line=dict(color='#c9d1d9', width=1.5),
))

fig.add_trace(go.Scatter(
    x=x_axis, y=step1['q50'][:n_plot].values,
    name='Forecast (q50)', line=dict(color='#58a6ff', width=2),
))

fig.update_layout(
    title=f"TFT Forecast | {cfg.target.upper()} | MAE={metrics['MAE']:.0f} | MAPE={metrics['MAPE_%']:.2f}%",
    paper_bgcolor='#0d1117', plot_bgcolor='#161b22',
    font=dict(color='#8b949e'), hovermode='x unified', height=500,
    xaxis=dict(title='Test Step', gridcolor='#21262d'),
    yaxis=dict(title=cfg.target.capitalize(), gridcolor='#21262d'),
    legend=dict(orientation='h', y=1.02, x=1, xanchor='right'),
)
fig.show()

plot_path = os.path.join(OUTPUT_DIR, 'forecast_plot.html')
fig.write_html(plot_path)
print(f'Plot saved -> {plot_path}')

Plot saved -> c:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection\tft_output\forecast_plot.html


## 🔎 Section 11 — Variable Importance (Optional)

In [55]:
try:
    importance = forecaster.get_variable_importance()
    if importance:
        for group, vals in importance.items():
            print(f'\n{group}:')
            if isinstance(vals, dict):
                for name, score in sorted(vals.items(), key=lambda x: x[1], reverse=True)[:10]:
                    print(f'  {name:<35}: {score:.4f}')
    else:
        print('Variable importance khong kha dung')
except Exception as e:
    print(f'Bo qua Variable Importance: {e}')

10:22:54 [WARNING] Variable importance thất bại: tuple indices must be integers or slices, not str


Variable importance khong kha dung


---
## ✅ Xong!

| File | Mô tả |
|------|--------|
| `tft_output/tft_results.json` | Metrics + full quantile predictions |
| `tft_output/checkpoints/` | Model checkpoints |
| `tft_output/forecast_plot.html` | Interactive Plotly chart |